# FEEDIT 브랜드 단위 Product 승격 + 매핑

브랜드명 하나만 바꿔가며 실행하는 운영용 노트북.

순서:
1. 무신사 ProductSource 전체를 Product로 승격하고 자기 자신을 연결
2. 나머지 플랫폼 전체(무신사 USED 포함)를 `style_no` exact로 1차 매핑
3. 아직 미매핑인 상품 중 USED를 제외하고 기존 `ProductMatchingPipeline`의 `EXACT`만 2차 매핑
4. 최종 미매핑을 플랫폼별로 확인

안전 원칙:
- 이미 `product_id`가 있는 ProductSource는 건드리지 않음
- 무신사 승격은 재실행 시 이미 연결된 건 건너뜀
- style_no가 둘 이상의 Product에 걸리면 자동 매핑하지 않음
- matching pipeline은 `EXACT`만 적용
- fuzzy / similarity는 자동 적용하지 않음


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\FEEDIT-DATA-CRAWLER")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
from django.apps import apps

if not apps.ready:
    django.setup()

print("Django ready")
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from collections import Counter, defaultdict

from django.db import transaction
from django.db.models import Q

from core.models import Brand, Product, ProductSource
from pipeline.product_matching.pipeline import ProductMatchingPipeline

print("Imports ready")


## 0. 브랜드명만 입력

평소에는 이 셀만 바꾸고 아래로 순서대로 실행.
`DRY_RUN=True`면 DB를 변경하지 않고 대상만 확인 가능.


In [ ]:
BRAND_NAME = "나이키"
DRY_RUN = True
TOP_K = 5

print("BRAND_NAME:", BRAND_NAME)
print("DRY_RUN   :", DRY_RUN)


In [ ]:
brand_name = str(BRAND_NAME or "").strip()

exact = Brand.objects.filter(name__iexact=brand_name).order_by("id")

if exact.count() != 1:
    print("정확 일치:", exact.count())
    print("후보:")
    for row in (
        Brand.objects
        .filter(
            Q(name__icontains=brand_name)
            | Q(english_name__icontains=brand_name)
        )
        .order_by("name", "id")[:30]
    ):
        print(
            row.id,
            "|",
            row.name,
            "|",
            getattr(row, "english_name", None),
        )
    raise ValueError("BRAND_NAME을 정확한 FEEDIT 브랜드명으로 입력하세요.")

brand = exact.first()

print("=" * 100)
print("BRAND ID :", brand.id)
print("NAME     :", brand.name)
print("ENGLISH  :", getattr(brand, "english_name", None))
print("=" * 100)


## 현재 브랜드 전체 현황

`source_brand__brand_id` 기준으로 이 브랜드에 귀속된 ProductSource를 모두 봄.


In [ ]:
brand_sources = (
    ProductSource.objects
    .filter(source_brand__brand_id=brand.id)
    .select_related("source", "source_brand", "product")
    .order_by("source_id", "id")
)

counts = Counter(
    (ps.source.code if ps.source_id else "UNKNOWN")
    for ps in brand_sources
)

print("Product        :", Product.objects.filter(brand_id=brand.id).count())
print("ProductSource  :", brand_sources.count())
print("Mapped         :", brand_sources.filter(product_id__isnull=False).count())
print("Unmapped       :", brand_sources.filter(product_id__isnull=True).count())
print()
print("[플랫폼별]")
for source_code, count in sorted(counts.items()):
    print(f"{source_code:20} {count}")


# STEP 1. 무신사 전체 Product 승격

기준:
- `source.code == musinsa`
- 아직 `product_id`가 없는 ProductSource만 신규 Product 생성
- Product 생성 후 해당 무신사 ProductSource를 즉시 연결
- 이미 연결된 무신사 ProductSource는 건너뜀

Product 필드는 현재 모델에 존재하는 필드만 사용:
- `brand`
- `canonical_name`
- `normalized_name`
- `product_code`

`product_code`는 우선 `ProductSource.style_no`, 없으면 `attributes.normalized.meta.product_code`를 사용.


In [ ]:
def model_has_field(model, field_name):
    try:
        model._meta.get_field(field_name)
        return True
    except Exception:
        return False


def clean(value):
    value = str(value or "").strip()
    return value or None


def get_ps_product_code(ps):
    style_no = clean(getattr(ps, "style_no", None))
    if style_no:
        return style_no

    attrs = getattr(ps, "attributes", None) or {}
    normalized = attrs.get("normalized") or {}
    meta = normalized.get("meta") or {}

    return clean(meta.get("product_code"))


def build_product_kwargs(ps, brand):
    raw_name = clean(getattr(ps, "source_name", None))
    normalized_name = clean(getattr(ps, "normalized_name", None))
    product_code = get_ps_product_code(ps)

    kwargs = {}

    if model_has_field(Product, "brand"):
        kwargs["brand"] = brand

    if model_has_field(Product, "canonical_name"):
        kwargs["canonical_name"] = normalized_name or raw_name or f"ProductSource {ps.id}"

    if model_has_field(Product, "normalized_name"):
        kwargs["normalized_name"] = normalized_name or raw_name

    if model_has_field(Product, "product_code"):
        kwargs["product_code"] = product_code

    return kwargs


musinsa_sources = list(
    brand_sources
    .filter(source__code__iexact="musinsa")
    .order_by("id")
)

already_promoted = [
    ps for ps in musinsa_sources
    if ps.product_id is not None
]

promotion_targets = [
    ps for ps in musinsa_sources
    if ps.product_id is None
]

print("=" * 100)
print("STEP 1 무신사 승격 대상")
print("=" * 100)
print("무신사 전체 :", len(musinsa_sources))
print("이미 연결   :", len(already_promoted))
print("신규 승격   :", len(promotion_targets))

for ps in promotion_targets[:30]:
    print(
        "PS",
        ps.id,
        "| STYLE:",
        getattr(ps, "style_no", None),
        "| NAME:",
        getattr(ps, "normalized_name", None)
        or getattr(ps, "source_name", None),
    )


In [ ]:
promotion_created = 0
promotion_skipped = 0
promotion_errors = []

if DRY_RUN:
    print("DRY_RUN=True: 승격하지 않음")

else:
    for index, ps in enumerate(promotion_targets, start=1):
        try:
            with transaction.atomic():
                locked = (
                    ProductSource.objects
                    .select_for_update()
                    .get(id=ps.id)
                )

                if locked.product_id is not None:
                    promotion_skipped += 1
                    continue

                kwargs = build_product_kwargs(
                    locked,
                    brand,
                )

                product = Product.objects.create(
                    **kwargs
                )

                locked.product_id = product.id
                locked.save(update_fields=["product"])

                promotion_created += 1

                print(
                    f"[{index}/{len(promotion_targets)}]",
                    "PS",
                    locked.id,
                    "-> PRODUCT",
                    product.id,
                    "|",
                    kwargs.get("product_code"),
                    "|",
                    kwargs.get("normalized_name")
                    or kwargs.get("canonical_name"),
                )

        except Exception as exc:
            promotion_errors.append({
                "product_source_id": ps.id,
                "error_type": type(exc).__name__,
                "error": str(exc),
            })
            print(
                "FAILED",
                ps.id,
                type(exc).__name__,
                str(exc),
            )

    print()
    print("CREATED:", promotion_created)
    print("SKIPPED:", promotion_skipped)
    print("FAILED :", len(promotion_errors))


# STEP 2. 나머지 플랫폼 style_no EXACT

대상:
- 현재 브랜드의 ProductSource
- `musinsa` 제외
- `musinsa_used` 포함
- 아직 미매핑인 상품

매칭:
- ProductSource.style_no
- 기준 Product에 연결된 무신사 ProductSource.style_no

동일 style_no가 정확히 하나의 Product로 결정될 때만 자동 매핑.
둘 이상의 Product가 같은 style_no를 가지면 `AMBIGUOUS`.


In [ ]:
def norm_style(value):
    value = str(value or "").strip()
    return value.casefold() if value else None


musinsa_master_sources = (
    ProductSource.objects
    .filter(
        product__brand_id=brand.id,
        source__code__iexact="musinsa",
        product_id__isnull=False,
    )
    .only("id", "product_id", "style_no")
)

style_to_products = defaultdict(set)

for ps in musinsa_master_sources:
    key = norm_style(ps.style_no)
    if key:
        style_to_products[key].add(ps.product_id)

style_unique = {
    key: next(iter(product_ids))
    for key, product_ids in style_to_products.items()
    if len(product_ids) == 1
}

style_ambiguous_master = {
    key: sorted(product_ids)
    for key, product_ids in style_to_products.items()
    if len(product_ids) > 1
}

style_targets = list(
    ProductSource.objects
    .filter(
        source_brand__brand_id=brand.id,
        product_id__isnull=True,
    )
    .exclude(source__code__iexact="musinsa")
    .select_related("source")
    .order_by("source_id", "id")
)

print("=" * 100)
print("STEP 2 STYLE NO EXACT")
print("=" * 100)
print("기준 style_no unique   :", len(style_unique))
print("기준 style_no ambiguous:", len(style_ambiguous_master))
print("검사 대상              :", len(style_targets))


In [ ]:
style_mapped = 0
style_no_value = 0
style_not_found = 0
style_ambiguous = []
style_errors = []

for index, ps in enumerate(style_targets, start=1):
    key = norm_style(ps.style_no)

    if not key:
        style_no_value += 1
        continue

    if key in style_ambiguous_master:
        style_ambiguous.append({
            "product_source_id": ps.id,
            "style_no": ps.style_no,
            "product_ids": style_ambiguous_master[key],
        })
        continue

    product_id = style_unique.get(key)

    if product_id is None:
        style_not_found += 1
        continue

    if DRY_RUN:
        print(
            "MATCH",
            ps.id,
            "|",
            ps.source.code,
            "| STYLE:",
            ps.style_no,
            "-> PRODUCT",
            product_id,
        )
        style_mapped += 1
        continue

    try:
        with transaction.atomic():
            locked = (
                ProductSource.objects
                .select_for_update()
                .get(id=ps.id)
            )

            if locked.product_id is not None:
                continue

            locked.product_id = product_id
            locked.save(update_fields=["product"])

            style_mapped += 1

    except Exception as exc:
        style_errors.append({
            "product_source_id": ps.id,
            "error_type": type(exc).__name__,
            "error": str(exc),
        })

print()
print("STYLE EXACT 매핑 :", style_mapped)
print("STYLE 없음       :", style_no_value)
print("기준 불일치      :", style_not_found)
print("AMBIGUOUS        :", len(style_ambiguous))
print("FAILED           :", len(style_errors))

if style_ambiguous:
    print()
    print("[AMBIGUOUS]")
    for row in style_ambiguous[:30]:
        print(row)


# STEP 3. USED 제외 ProductMatchingPipeline EXACT

대상은 기존 pipeline 정책 그대로:
- 미매핑 ProductSource
- `musinsa_used` / `musinsa-used` 제외

자동 적용 조건:
- Top-K 후보 중 `evidence.details["method"] == "EXACT"`
- EXACT 후보가 정확히 하나의 Product일 때만 매핑
- TOKEN_JACCARD / SEQUENCE는 절대 자동 매핑하지 않음


In [ ]:
matching_pipeline = ProductMatchingPipeline(
    brand_id=brand.id,
    top_k=TOP_K,
)

matching_result = matching_pipeline.run()

exact_plan = []
exact_ambiguous = []
non_exact = []

for match in matching_result.matches:
    exact_candidates = [
        candidate
        for candidate in match.candidates
        if (
            candidate.evidence
            and candidate.evidence.details.get("method") == "EXACT"
        )
    ]

    exact_product_ids = sorted({
        candidate.product_id
        for candidate in exact_candidates
    })

    if len(exact_product_ids) == 1:
        exact_plan.append({
            "product_source_id": match.product_source_id,
            "source_code": match.source_code,
            "source_name": match.source_name,
            "normalized_name": match.normalized_name,
            "product_id": exact_product_ids[0],
        })

    elif len(exact_product_ids) > 1:
        exact_ambiguous.append({
            "product_source_id": match.product_source_id,
            "product_ids": exact_product_ids,
        })

    else:
        non_exact.append(match)

print("=" * 100)
print("STEP 3 PIPELINE EXACT")
print("=" * 100)
print("Pipeline 미매핑 대상 :", matching_result.unmapped_count)
print("EXACT 매핑 예정      :", len(exact_plan))
print("EXACT ambiguous      :", len(exact_ambiguous))
print("EXACT 없음           :", len(non_exact))

print()
for row in exact_plan[:50]:
    print(
        "PS",
        row["product_source_id"],
        "|",
        row["source_code"],
        "|",
        row["normalized_name"] or row["source_name"],
        "-> PRODUCT",
        row["product_id"],
    )


In [ ]:
exact_mapped = 0
exact_skipped = 0
exact_errors = []

if DRY_RUN:
    exact_mapped = len(exact_plan)
    print("DRY_RUN=True: EXACT 매핑 예정 건만 표시")

else:
    for index, row in enumerate(exact_plan, start=1):
        try:
            with transaction.atomic():
                ps = (
                    ProductSource.objects
                    .select_for_update()
                    .get(id=row["product_source_id"])
                )

                if ps.product_id is not None:
                    exact_skipped += 1
                    continue

                product_exists = Product.objects.filter(
                    id=row["product_id"],
                    brand_id=brand.id,
                ).exists()

                if not product_exists:
                    raise ValueError(
                        f"Product가 현재 브랜드에 속하지 않습니다: {row['product_id']}"
                    )

                ps.product_id = row["product_id"]
                ps.save(update_fields=["product"])

                exact_mapped += 1

        except Exception as exc:
            exact_errors.append({
                "product_source_id": row["product_source_id"],
                "error_type": type(exc).__name__,
                "error": str(exc),
            })

    print("EXACT MAPPED :", exact_mapped)
    print("SKIPPED      :", exact_skipped)
    print("FAILED       :", len(exact_errors))


# STEP 4. 최종 결과

`DRY_RUN=False`로 실제 적용한 뒤 실행하면 최종 DB 상태를 보여줌.
남은 미매핑은 자동으로 더 건드리지 않음.


In [ ]:
final_sources = (
    ProductSource.objects
    .filter(source_brand__brand_id=brand.id)
    .select_related("source")
)

final_products = Product.objects.filter(
    brand_id=brand.id
)

final_mapped = final_sources.filter(
    product_id__isnull=False
)

final_unmapped = final_sources.filter(
    product_id__isnull=True
)

unmapped_by_source = Counter(
    ps.source.code if ps.source_id else "UNKNOWN"
    for ps in final_unmapped
)

print("=" * 100)
print("최종 결과")
print("=" * 100)
print("BRAND             :", brand.name)
print("Product           :", final_products.count())
print("ProductSource     :", final_sources.count())
print("Mapped PS         :", final_mapped.count())
print("Unmapped PS       :", final_unmapped.count())

print()
print("[최종 미매핑 플랫폼별]")
for source_code, count in sorted(unmapped_by_source.items()):
    print(f"{source_code:20} {count}")

print()
print("[최종 미매핑 샘플]")
for ps in (
    final_unmapped
    .select_related("source")
    .order_by("source_id", "id")[:100]
):
    print(
        "PS",
        ps.id,
        "|",
        ps.source.code if ps.source_id else None,
        "| STYLE:",
        getattr(ps, "style_no", None),
        "|",
        getattr(ps, "normalized_name", None)
        or getattr(ps, "source_name", None),
    )


## 실제 운영 순서

처음에는:

```python
BRAND_NAME = "원하는 브랜드"
DRY_RUN = True
```

로 전체 계획 확인.

문제 없으면:

```python
DRY_RUN = False
```

로 바꾸고 STEP 1 → STEP 2 → STEP 3 → STEP 4 순서대로 실행.

다음 브랜드로 넘어갈 때 `BRAND_NAME`만 변경하면 됨.
